# PAD Prediction Model v2.0

Predicting Peripheral Artery Disease (PAD) from MIMIC-IV clinical data using multiple ML approaches.

**Pipeline overview:**
1. Load and merge MIMIC-IV tables (admissions, patients, diagnoses, labs, prescriptions)
2. Engineer clinical features — demographics, lab results, comorbidities, medications
3. Train and compare six classifiers
4. Evaluate with AUC, confusion matrix, and ROC curves

In [ ]:
import pandas as pd
import numpy as np
import random
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix

from xgboost import XGBClassifier
import lightgbm as lgb

warnings.filterwarnings('ignore')

## Configuration

In [ ]:
drive.mount('/content/drive')

GDRIVE_MIMIC_PATH = '/content/drive/MyDrive/mimic_data/'

PAD_CODES = {
    'icd9': ['4439', '44389', '44381', '44022', '44029', '4408', '4409'],
    'icd10': ['I739', 'I7021', 'I7022', 'I7029', 'I708', 'I709', 'I7391', 'I7399']
}

COMORBIDITY_CODES = {
    'has_diabetes': {
        'icd9': ['250'],
        'icd10': ['E10', 'E11', 'E13']
    },
    'has_hypertension': {
        'icd9': ['401', '402', '403', '404', '405'],
        'icd10': ['I10', 'I11', 'I12', 'I13', 'I15']
    },
    'has_heart_disease': {
        'icd9': ['410', '411', '412', '413', '414'],
        'icd10': ['I20', 'I21', 'I22', 'I23', 'I24', 'I25']
    },
    'has_stroke_history': {
        'icd9': ['430', '431', '432', '433', '434', '436'],
        'icd10': ['I60', 'I61', 'I62', 'I63', 'I64']
    }
}

LAB_TESTS = ['Cholesterol, Total', 'Glucose', 'Creatinine', 'Hemoglobin', 'Platelet Count']

MEDICATION_KEYWORDS = {
    'is_on_statin': 'statin|atorvastatin|rosuvastatin|simvastatin',
    'is_on_antiplatelet': 'aspirin|clopidogrel|plavix'
}

## Data Loading

In [ ]:
admissions_df = pd.read_csv(GDRIVE_MIMIC_PATH + 'admissions.csv.gz', compression='gzip')
patients_df = pd.read_csv(GDRIVE_MIMIC_PATH + 'patients.csv.gz', compression='gzip')
diagnoses_df = pd.read_csv(GDRIVE_MIMIC_PATH + 'diagnoses_icd.csv.gz', compression='gzip')
d_labitems_df = pd.read_csv(GDRIVE_MIMIC_PATH + 'd_labitems.csv.gz', compression='gzip')

print(f"Admissions: {admissions_df.shape}")
print(f"Patients: {patients_df.shape}")
print(f"Diagnoses: {diagnoses_df.shape}")
print(f"Lab items dict: {d_labitems_df.shape}")

## Cohort Identification

In [ ]:
pad_diagnoses = diagnoses_df[
    (diagnoses_df['icd_code'].str.startswith(tuple(PAD_CODES['icd10']), na=False) & (diagnoses_df['icd_version'] == 10)) |
    (diagnoses_df['icd_code'].str.startswith(tuple(PAD_CODES['icd9']), na=False) & (diagnoses_df['icd_version'] == 9))
]

pad_admission_ids = pad_diagnoses['hadm_id'].unique()
all_admission_ids = admissions_df['hadm_id'].unique()
non_pad_ids = [hadm_id for hadm_id in all_admission_ids if hadm_id not in pad_admission_ids]

# FIX #1: Set random seed before sampling so the cohort is reproducible across runs
random.seed(42)
control_admission_ids = random.sample(non_pad_ids, len(pad_admission_ids))

pad_df = pd.DataFrame({'hadm_id': pad_admission_ids, 'had_pad': 1})
control_df = pd.DataFrame({'hadm_id': control_admission_ids, 'had_pad': 0})
master_df = pd.concat([pad_df, control_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"PAD cases: {len(pad_admission_ids)}")
print(f"Control cases: {len(control_admission_ids)}")
print(f"Total cohort: {len(master_df)}")

## Feature Engineering

### Demographics

In [ ]:
master_df = pd.merge(master_df, admissions_df[['hadm_id', 'subject_id', 'admittime']], on='hadm_id', how='left')
master_df = pd.merge(master_df, patients_df[['subject_id', 'gender', 'anchor_age', 'anchor_year']], on='subject_id', how='left')
master_df['admittime'] = pd.to_datetime(master_df['admittime'])
master_df['age_at_admission'] = master_df['anchor_age'] + (master_df['admittime'].dt.year - master_df['anchor_year'])
master_df.drop(columns=['anchor_age', 'anchor_year'], inplace=True)

# FIX #2 (temporal leakage): Rename admittime to admittime_index and keep it.
# It will be used to restrict comorbidities/meds to pre-index time, then dropped before modelling.
master_df.rename(columns={'admittime': 'admittime_index'}, inplace=True)

master_df['age_at_admission'].fillna(master_df['age_at_admission'].median(), inplace=True)
subject_ids_we_want = master_df['subject_id'].unique()

# Build a time-stamped view of all admissions for our subjects (used in comorbidity/med temporal filtering)
all_subj_admissions = admissions_df[
    admissions_df['subject_id'].isin(subject_ids_we_want)
][['subject_id', 'hadm_id', 'admittime']].copy()
all_subj_admissions['admittime'] = pd.to_datetime(all_subj_admissions['admittime'])

master_df[['gender', 'age_at_admission']].describe()

### Lab Results

In [ ]:
target_lab_ids = d_labitems_df[d_labitems_df['label'].isin(LAB_TESTS)][['itemid', 'label']]
target_itemid_list = target_lab_ids['itemid'].tolist()

chunk_size = 1_000_000
list_of_lab_chunks = []
lab_chunk_iterator = pd.read_csv(
    GDRIVE_MIMIC_PATH + 'labevents.csv.gz',
    compression='gzip', chunksize=chunk_size, low_memory=False
)

for chunk in lab_chunk_iterator:
    chunk_filtered = chunk[
        chunk['subject_id'].isin(subject_ids_we_want) &
        chunk['itemid'].isin(target_itemid_list)
    ]
    if not chunk_filtered.empty:
        list_of_lab_chunks.append(chunk_filtered)

if list_of_lab_chunks:
    filtered_labs = pd.concat(list_of_lab_chunks).dropna(subset=['valuenum'])

    # FIX #3: Map itemid -> label BEFORE aggregating so that multiple itemids sharing the same
    # label (e.g. point-of-care vs lab Glucose) are averaged together rather than creating
    # duplicate columns that get silently dropped later.
    filtered_labs = pd.merge(filtered_labs, target_lab_ids, on='itemid', how='inner')
    aggregated_labs = filtered_labs.groupby(['hadm_id', 'label'])['valuenum'].mean().reset_index()
    pivoted_labs = aggregated_labs.pivot(index='hadm_id', columns='label', values='valuenum').reset_index()
    pivoted_labs.columns.name = None
    master_df = pd.merge(master_df, pivoted_labs, on='hadm_id', how='left')

print("Lab features added:", [c for c in master_df.columns if c in LAB_TESTS])

### Comorbidities

In [ ]:
# FIX #4 (temporal leakage): Only flag a comorbidity if it was diagnosed in an admission
# that occurred AT OR BEFORE the index admission for that patient.
# Without this fix, future diagnoses (post-index) would leak into the feature set.

# Join all diagnoses with their admission timestamps
diagnoses_timed = pd.merge(
    diagnoses_df[['subject_id', 'hadm_id', 'icd_code', 'icd_version']],
    all_subj_admissions[['hadm_id', 'admittime']].rename(columns={'admittime': 'diag_admittime'}),
    on='hadm_id', how='inner'
)

index_times = master_df[['hadm_id', 'subject_id', 'admittime_index']]

for disease, codes in COMORBIDITY_CODES.items():
    disease_diag = diagnoses_timed[
        (diagnoses_timed['icd_code'].str.startswith(tuple(codes['icd10']), na=False) & (diagnoses_timed['icd_version'] == 10)) |
        (diagnoses_timed['icd_code'].str.startswith(tuple(codes['icd9']), na=False) & (diagnoses_timed['icd_version'] == 9))
    ][['subject_id', 'diag_admittime']].drop_duplicates()

    # For each index admission, check if any matching diagnosis occurred at or before it
    merged = pd.merge(index_times, disease_diag, on='subject_id', how='left')
    valid_hadm_ids = set(
        merged[merged['diag_admittime'] <= merged['admittime_index']]['hadm_id'].dropna().unique()
    )
    master_df[disease] = master_df['hadm_id'].isin(valid_hadm_ids).astype(int)

master_df[list(COMORBIDITY_CODES.keys())].sum()

### Medications

In [ ]:
# FIX #5 (temporal leakage): Only flag a medication if it was prescribed in an admission
# at or before the index admission. Collects all filtered prescription rows first,
# then applies the temporal constraint in a single vectorised step.

chunk_size = 1_000_000
presc_chunks = []

prescriptions_iterator = pd.read_csv(
    GDRIVE_MIMIC_PATH + 'prescriptions.csv.gz',
    compression='gzip', chunksize=chunk_size, low_memory=False
)

for chunk in prescriptions_iterator:
    chunk_filtered = chunk[chunk['subject_id'].isin(subject_ids_we_want)][['subject_id', 'hadm_id', 'drug']]
    if not chunk_filtered.empty:
        presc_chunks.append(chunk_filtered)

if presc_chunks:
    all_prescriptions = pd.concat(presc_chunks)

    # Attach prescription admission timestamps for temporal filtering
    presc_timed = pd.merge(
        all_prescriptions,
        all_subj_admissions[['hadm_id', 'admittime']].rename(columns={'admittime': 'presc_admittime'}),
        on='hadm_id', how='inner'
    )

    index_times = master_df[['hadm_id', 'subject_id', 'admittime_index']]

    for med_class, keywords in MEDICATION_KEYWORDS.items():
        med_df = presc_timed[
            presc_timed['drug'].str.contains(keywords, case=False, na=False)
        ][['subject_id', 'presc_admittime']].drop_duplicates()

        merged = pd.merge(index_times, med_df, on='subject_id', how='left')
        valid_hadm_ids = set(
            merged[merged['presc_admittime'] <= merged['admittime_index']]['hadm_id'].dropna().unique()
        )
        master_df[med_class] = master_df['hadm_id'].isin(valid_hadm_ids).astype(int)

master_df[list(MEDICATION_KEYWORDS.keys())].sum()

### Final Cleanup

In [ ]:
master_df['gender'] = master_df['gender'].apply(lambda x: 1 if x == 'M' else 0)
master_df.rename(columns={
    'Cholesterol, Total': 'cholesterol',
    'Glucose': 'glucose',
    'Creatinine': 'creatinine',
    'Hemoglobin': 'hemoglobin',
    'Platelet Count': 'platelet_count'
}, inplace=True)
master_df = master_df.loc[:, ~master_df.columns.duplicated()].copy()

# Drop identifiers and the temporal anchor column (used only for leakage prevention above)
final_model_df = master_df.drop(columns=['hadm_id', 'subject_id', 'admittime_index'])
print(f"Final dataset: {final_model_df.shape}")
final_model_df.info()

### Save Processed Dataset

In [ ]:
filename = 'pad_model_dataset.csv'
final_model_df.to_csv(filename, index=False)
print(f"Saved to {filename}")

try:
    from google.colab import files
    files.download(filename)
except ImportError:
    pass

## Model Training & Comparison

In [ ]:
y = final_model_df['had_pad']
X = final_model_df.drop('had_pad', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()
X_train_processed = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_processed = scaler.transform(imputer.transform(X_test))

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# FIX #6: MLPClassifier does not support class_weight, so we use compute_sample_weight
# to give each training sample a weight that corrects for class imbalance.
# FIX #7: LightGBM now gets class_weight='balanced' to match the other classifiers.
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    "SVM": SVC(probability=True, class_weight='balanced', random_state=42),
    "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', scale_pos_weight=scale_pos_weight, random_state=42),
    "LightGBM": lgb.LGBMClassifier(class_weight='balanced', random_state=42)
}

In [ ]:
results = {}
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    # Pass sample_weights for MLP (which lacks class_weight parameter);
    # all other models use their built-in class_weight / scale_pos_weight instead.
    if name == 'Neural Network (MLP)':
        model.fit(X_train_processed, y_train, sample_weight=sample_weights)
    else:
        model.fit(X_train_processed, y_train)
    trained_models[name] = model

    y_pred = model.predict(X_test_processed)
    y_pred_proba = model.predict_proba(X_test_processed)[:, 1]

    auc = roc_auc_score(y_test, y_pred_proba)
    report = classification_report(y_test, y_pred, output_dict=True)

    results[name] = {
        "AUC": auc,
        "Accuracy": report['accuracy'],
        "Recall (PAD)": report['1']['recall'],
        "Precision (PAD)": report['1']['precision']
    }

## Results

In [ ]:
results_df = pd.DataFrame(results).T
results_df['AUC'] = results_df['AUC'].apply(lambda x: f"{x:.4f}")
results_df['Accuracy'] = results_df['Accuracy'].apply(lambda x: f"{x*100:.2f}%")
results_df['Recall (PAD)'] = results_df['Recall (PAD)'].apply(lambda x: f"{x*100:.2f}%")
results_df['Precision (PAD)'] = results_df['Precision (PAD)'].apply(lambda x: f"{x*100:.2f}%")

results_df

## Best Model Analysis

In [ ]:
best_model_name = max(results, key=lambda name: results[name]['AUC'])
best_model = trained_models[best_model_name]
print(f"Best model: {best_model_name}")

y_pred_best = best_model.predict(X_test_processed)
y_pred_proba_best = best_model.predict_proba(X_test_processed)[:, 1]
auc_best = roc_auc_score(y_test, y_pred_proba_best)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No PAD', 'Has PAD'],
            yticklabels=['No PAD', 'Has PAD'], ax=axes[0])
axes[0].set_title(f'Confusion Matrix — {best_model_name}')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(y_test, y_pred_proba_best)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {auc_best:.2f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC Curve — {best_model_name}')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()